# Transformer Block 与 Language Model

本节讨论 Decoder-only Transformer（因果自注意力）。语言模型的目标是根据已有 Token 预测下一个 Token。Transformer Block 负责学习上下文特征，LM Head 负责利用这些特征计算预测分数。

## 1. 整体结构

```text
原始文本
  ↓ Tokenizer：切分文本，取得 Token ID
Token IDs                (B, L)
  ↓ Embedding：根据 ID 查找词嵌入向量
初始特征 X₀              (B, L, d_model)
  ↓ Transformer Block × N：信息交互与非线性特征学习
  ↓ Final RMSNorm：控制最终特征尺度
上下文特征 H             (B, L, d_model)
  ↓ LM Head：计算每个位置对所有候选 Token 的预测分数
Logits Z                 (B, L, V)
```

| 符号 | 含义 |
|---|---|
| $B$ | 批次大小 |
| $L$ | 当前输入序列的 Token 数量 |
| $d_{\text{model}}$ | 每个 Token 的特征维度 |
| $V$ | 词表中不同 Token 的数量（vocab_size） |
| $N$ | Transformer Block 的数量 |

本节重点是 LM Head 的两种参数设计：**共享权重（Weight Tying）与不共享权重（Untied Weights）**。两者的计算方式相同，区别在于用于预测的权重矩阵是否复用输入词嵌入矩阵。

## 2. Tokenizer 与 Token Embedding

Tokenizer 将文本切分为 Token，再根据词表取得整数 ID。例如一个简化的词表：

| Token | I | like | apples | bananas | eat | EOS |
|---|---:|---:|---:|---:|---:|---:|
| ID | 0 | 1 | 2 | 3 | 4 | 5 |

输入 `I like` 对应 `[0, 1]`，此时 $L=2$，而整个词表大小 $V=6$。Token ID 只是查表索引，不表示语义大小。实际 Tokenizer 也可以将单词拆成多个子词 Token。

为了为每个 Token 建立可学习的特征表示，定义词嵌入矩阵：

$$
E=\begin{bmatrix} e_0^\top\\ e_1^\top\\ \vdots\\ e_{V-1}^\top\end{bmatrix}\in\mathbb R^{V\times d_{\text{model}}}.
$$

其中 $e_j$ 是词表中第 $j$ 个 Token 的词嵌入向量。输入 `[0, 1]` 时，只取出 $E$ 中对应的两行。假设 $d_{\text{model}}=4$：

$$X_0=\begin{bmatrix}e_0^\top\\e_1^\top\end{bmatrix}\in\mathbb R^{2\times4}.$$

$E$ 是整个词表的可学习参数矩阵，形状为 $(V,d_{\text{model}})$；$X_0$ 是当前输入的特征矩阵，省略批次维度后形状为 $(L,d_{\text{model}})$。词表大小 $V$ 并不等于 $L\times d_{\text{model}}$。

```python
# 内部封装可学习的词嵌入矩阵 E：(V, d_model)
self.embedding = nn.Embedding(vocab_size, d_model)

# 根据 Token ID 查表，取出对应的向量
x = self.embedding(token_ids)  # (B, L, d_model)
```

查表等价于 One-Hot 向量与 $E$ 相乘，但无需实际构造 One-Hot 向量。从零训练时，$E$ 与其余网络参数一起初始化并联合训练，不需要提前训练好词嵌入。

## 3. Transformer Block 与 Final Norm

初始词嵌入只取决于 Token ID，没有结合当前序列的上下文。因此，通过 Attention 进行 Token 之间的信息交互，再通过 FFN 对每个 Token 内部的特征进行非线性学习。

采用 Pre-Norm 时，一个 Block 的计算过程为：

$$T=X+\operatorname{Attention}(\operatorname{RMSNorm}_1(X)),$$
$$Y=T+\operatorname{FFN}(\operatorname{RMSNorm}_2(T)).$$

这里使用带 RoPE 的因果多头自注意力以及 SwiGLU FFN。因果掩码限制每个位置只能访问当前及之前的 Token。RMSNorm 控制子层输入尺度，残差连接保留恒等梯度传播路径。

```python
# Transformer Block 内部：先进行 Token 之间的信息交互
x = x + self.attention(self.norm1(x), is_causal=True)

# 再对每个 Token 内部的特征进行非线性变换
x = x + self.ffn(self.norm2(x))
```

每个 Block 保持输入输出形状一致，便于堆叠。Pre-Norm 不直接归一化残差相加后的输出，因此这里在最后一个 Block 之后再使用 Final RMSNorm。

```python
# N 个独立的 Block，ModuleList 用于注册其中的网络参数
self.blocks = nn.ModuleList([
    TransformerBlock(d_model, num_heads)
    for _ in range(num_layers)
])
self.final_norm = RMSNorm(d_model)

for block in self.blocks:
    x = block(x)
H = self.final_norm(x)  # (B, L, d_model)
```

省略批次维度，记 $H=f_\theta(X_0)\in\mathbb R^{L\times d_{\text{model}}}$，其中 $f_\theta$ 包含所有 Block 和 Final Norm。$H$ 的第 $i$ 行是 $h_i^\top$，即第 $i$ 个位置的上下文特征。$h_i$ 是计算得到的向量，不是独立的可学习参数。

## 4. LM Head：共享权重与不共享权重（重点）

LM Head（Language Modeling Head）是语言模型输出预测层。LM 表示语言建模任务，Head 表示连接在主干网络之后、完成具体预测任务的模块。这里的 LM Head 本质上就是一个**不带偏置的线性层**。

经过 Transformer 后，每个位置已经得到上下文特征 $h_i$。为了预测下一个 Token，需要为整个词表中的每个候选 Token 计算一个分数。

### 4.1 共享权重（Weight Tying）

Embedding 已经为每个候选 Token 定义了一个可学习向量 $e_j$，因此可以直接复用这些向量，与当前上下文特征进行内积打分：

$$\boxed{z_{i,j}=h_i^\top e_j}$$

例如输入 `I like`，将 `like` 位置的上下文特征分别与 `apples`、`bananas` 等词表中所有候选 Token 的词嵌入向量计算内积。

为了同时计算所有位置与所有候选的分数，使用矩阵乘法：

$$\boxed{Z=HE^\top}$$

同一个矩阵 $E$ 既用于输入查表，又用于输出预测，因此称为 Weight Tying。无需额外学习一套输出权重。

```python
# E：(V, d_model)，H：(B, L, d_model)
# 直接复用 Embedding 的权重，计算候选 Token 的内积分数
logits = H @ self.embedding.weight.T  # (B, L, V)
```

### 4.2 不共享权重（Untied Weights）

另一种设计是**不复用 Embedding，而在最后一个 Block 和 Final Norm 之后使用独立的 Linear 层**。这样可以专门学习如何组合上下文特征内部的各个维度，以预测不同的 Token。

定义独立的可学习输出矩阵 $W\in\mathbb R^{V\times d_{\text{model}}}$。第 $j$ 行为 $w_j^\top$，用于计算候选 Token $j$ 的分数：

$$\boxed{z_{i,j}=h_i^\top w_j}$$

假设 $h_i=[a,b,c,d]^\top$，则：

$$z_{i,j}=aw_{j,1}+bw_{j,2}+cw_{j,3}+dw_{j,4}.$$

这就是 Linear 层内部的计算：对一个 Token 的特征进行线性加权组合。它不再进行 Token 之间的信息交互。同一个 $W$ 用于所有位置。

$$\boxed{Z=HW^\top}$$

```python
# 创建独立的输出线性层，内部封装 W：(V, d_model)
# bias=False：只执行 H @ W.T，不加偏置
self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

logits = self.lm_head(H)  # (B, L, V)
```

**Linear 层本身就是通过内积计算各输出分数的。** 因此，不共享不是在原有内积操作之前增加一个 Linear 层，而是使用独立矩阵 $W$ 替代原本用于输出打分的 $E$。

### 4.3 用一个数值例子理解区别

假设词表只有 A、B 两个 Token，特征维度为 2。已有词嵌入 $E$，当前上下文特征 $h_i$ 为：

$$E=\begin{bmatrix}1&0\\0&1\end{bmatrix},\qquad h_i=\begin{bmatrix}2\\1\end{bmatrix}.$$

共享权重时，直接使用 $E$ 中的两个向量。假设不共享时，独立学习得到的输出权重为：

$$W=\begin{bmatrix}0&1\\1&0\end{bmatrix}.$$

| 候选 Token | 共享：与 $e_j$ 做内积 | 不共享：与 $w_j$ 做内积 |
|---|---|---|
| A | $2\times1+1\times0=2$ | $2\times0+1\times1=1$ |
| B | $2\times0+1\times1=1$ | $2\times1+1\times0=2$ |
| Logits | $[2,1]$ | $[1,2]$ |
| Softmax 后的概率（约） | $[0.731,0.269]$ | $[0.269,0.731]$ |
| 选择概率最大者 | A | B |

不共享时，可以通过改变 $W$ 中的权重来改变输出打分，而不必同步修改 $E$ 中对应的输入向量。这就是独立 Linear 层的作用。

这里的数值仅用于说明两种打分方式的区别，并不说明哪种预测更好。实际训练中，$E$、Transformer 参数以及独立的 $W$ 都可以更新；共享模型也可以通过学习 $E$ 和上下文特征得到正确预测。

### 4.4 为什么有不共享权重的设计？

- **共享权重：** 复用已有的 Token 表示，让同一套参数兼顾输入查表与输出打分，减少参数量。
- **不共享权重：** 输入词嵌入用于为 Transformer 提供特征，而输出 Linear 层用于预测。两种用途不同，因此可以让输出层独立学习适合预测的特征加权方式，不必和输入词嵌入使用相同的权重。

不共享并不代表冻结 Embedding，也不需要分两阶段训练。预测损失仍然会经过 Transformer 回传到输入词嵌入。两种设计都合理，独立学习不保证效果一定更好。

| 对比 | 共享权重 | 不共享权重 |
|---|---|---|
| 输入 | 上下文特征 $H$ | 上下文特征 $H$ |
| 单个候选的分数 | $h_i^\top e_j$ | $h_i^\top w_j$ |
| 矩阵形式 | $HE^\top$ | $HW^\top$ |
| 权重来源 | 输入词嵌入矩阵 $E$ | 独立 Linear 权重 $W$ |
| 是否进行新的 Token 间交互 | 否 | 否 |
| 输入、输出两部分总权重参数量 | $Vd_{\text{model}}$ | $2Vd_{\text{model}}$ |
| 后续操作 | Softmax（得到概率） | Softmax（得到概率） |

### 4.5 统一的 PyTorch 写法

两种方式都可以用 `nn.Linear` 表示，只需在初始化时决定是否共享参数：

```python
self.embedding = nn.Embedding(vocab_size, d_model)
self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

# 共享：让 Linear 与 Embedding 引用同一个权重参数（不是复制）
# 不共享：不执行下面的赋值，保留 Linear 自己的独立参数 W
if tie_weights:
    self.lm_head.weight = self.embedding.weight

# 两种情况下，forward 都执行相同的内积计算
logits = self.lm_head(H)  # (B, L, V)
```

**共享和不共享不是两种不同的打分算法，而是同一个线性预测层的两种权重设计。**

## 5. Logits、Softmax 与下一 Token 预测

LM Head 输出的分数称为 Logits，形状为 $(B,L,V)$。其中 $Z$ 的最后一维包含一个位置对整个词表的预测分数。沿词表维度进行 Softmax，得到各候选 Token 的预测概率：

$$p_{i,j}=\frac{\exp(z_{i,j})}{\sum_{k=0}^{V-1}\exp(z_{i,k})}.$$

**训练阶段：** 假设完整序列为 `I like apples EOS`，将序列错开一个位置构造输入与标签。

| 输入位置 | I | like | apples |
|---|---|---|---|
| 可见上下文 | I | I like | I like apples |
| 真实下一 Token | like | apples | EOS |

因果自注意力保证每个位置看不到未来 Token，因此可以并行计算所有位置的预测分数，再利用交叉熵损失更新模型参数。`nn.CrossEntropyLoss` 内部包含 LogSoftmax，训练时直接传入 Logits 即可。

**推理阶段：** 使用最后一个位置的 Logits 预测下一个 Token，例如通过 Softmax 获得概率，再选取概率最大的候选（Greedy Decoding），或根据概率进行采样。若只选择最大值，也可以直接对 Logits 执行 `argmax`，因为 Softmax 不改变分数的排序。

```python
# 最后一个位置对整个词表的预测分数
next_logits = logits[:, -1, :]  # (B, V)

# 沿词表维度归一化，得到每个候选 Token 的概率
probs = torch.softmax(next_logits, dim=-1)

# 贪心解码：选择概率最大的 Token ID
next_token = probs.argmax(dim=-1)  # (B,)
```

### 总结

Attention 学习 Token 之间的关系，FFN 对每个 Token 内部的特征进行非线性变换，LM Head 则利用最终的上下文特征进行线性加权打分。LM Head 可以复用输入词嵌入矩阵（共享权重），也可以使用独立的 Linear 权重（不共享权重）。两种方式都通过内积得到 Logits，只是参数来源不同。
